# Is the cactus/PG imbalance *caused by* missingness?

**Created 2026-05-29.** The big question before we commit to any `cn` fix: is the
14.3× cactus/PG private-k-mer imbalance a **consequence of N-on dropping PG
missing-GT k-mers**, or **real biology** (cactus long-reads realize more rare
alleles)?

**The decisive trick — no rebuild, no REF-fabrication.** In any bubble where
*every* founder is called (no `./.`), N-on writes no N, drops nothing, and `cn`
is byte-identical to what an N-off build would produce. So **the imbalance
measured inside fully-called bubbles is pure biology** — missingness *cannot*
contribute there. We then stratify bubbles by PG missingness and watch the
imbalance:

- imbalance ~flat across missingness, large even at zero  → **biology**
- imbalance ~1× at zero missingness, growing with it      → **caused by missingness**

Uses only `data/cn_full_231_v3qc_v3/` (N-on) + the panel VCF.

> **Approximation (honest caveat):** we mark a founder "missing in bubble b" if
> it has *any* `./.` among b's variants. N-on actually only drops k-mers
> *overlapping* the specific missing variant, so in multi-variant bubbles this
> slightly over-attributes to missingness. It is **exact** for single-variant
> bubbles (most SVs) and **exact** for the zero-missingness stratum (nothing is
> dropped there) — which is the dispositive number.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
import pysam

ROOT = Path('/global/scratch/users/tbellg/kmate')
CN   = ROOT / 'data/cn_full_231_v3qc_v3/cn_Chr1.cn.npz'
META = ROOT / 'data/cn_full_231_v3qc_v3/cn_Chr1.meta.npz'
VCF  = ROOT / 'panel/pangenie_genotyping/data/v3qc_v3/founders_231_v3qc_v3.haploid.vcf.gz'
SPLIT= ROOT / 'data/founder_split_cactus_pg.json'
CHROM= 'Chr1'
PLOTDIR = ROOT/'notebooks/plots'; PLOTDIR.mkdir(exist_ok=True)

cn   = load_npz(CN).tocsr()
meta = np.load(META, allow_pickle=True)
founders = np.asarray(meta['founders']).astype(str)
F = len(founders)
bubble_id   = np.asarray(meta['bubble_id']).astype(np.int64)
bub_start   = np.asarray(meta['bubble_start']).astype(np.int64)
bub_end     = np.asarray(meta['bubble_end']).astype(np.int64)
n_bubbles   = len(bub_start)

with open(SPLIT) as f: split = json.load(f)
cset, pset = set(map(str, split['cactus'])), set(map(str, split['PG']))
is_cactus = np.array([f in cset for f in founders])
is_pg     = np.array([f in pset for f in founders])
n_cac, n_pg = int(is_cactus.sum()), int(is_pg.sum())
print(f'cn={cn.shape}, bubbles={n_bubbles:,}, cactus={n_cac}, PG={n_pg}')

# per-k-mer carrier counts
ac    = np.asarray(cn.sum(0)).flatten()
ac_c  = np.asarray(cn[is_cactus].sum(0)).flatten()
ac_p  = np.asarray(cn[is_pg].sum(0)).flatten()

In [ ]:
# --- Per-bubble missingness from the VCF (one pass, fetch per bubble) ---
# missing_in_bubble[b, f] = founder f has >=1 './.' among bubble b's variants.
vcf = pysam.VariantFile(str(VCF))
vsamples = list(vcf.header.samples)
# map cn founder order -> VCF column index
col = {s: i for i, s in enumerate(vsamples)}
order = np.array([col[f] for f in founders])  # reorder VCF gts -> cn founder order

miss_bub = np.zeros((n_bubbles, F), dtype=bool)
for b in range(n_bubbles):
    s, e = int(bub_start[b]), int(bub_end[b])
    try:
        recs = list(vcf.fetch(CHROM, max(0, s-1), e))
    except Exception:
        continue
    if not recs:
        continue
    m = np.zeros(F, dtype=bool)
    for rec in recs:
        gts = rec.samples  # indexable by sample name
        for j, s_name in enumerate(vsamples):
            gt = gts[s_name]['GT']
            if gt is None or len(gt) == 0 or gt[0] is None:
                m[col[s_name]] = True
    miss_bub[b] = m[order]
    if (b+1) % 20000 == 0:
        print(f'  bubbles {b+1:,}/{n_bubbles:,}')
print('done VCF pass')

# per-bubble side missingness fractions
pg_miss_frac  = miss_bub[:, is_pg].sum(1) / n_pg
cac_miss_frac = miss_bub[:, is_cactus].sum(1) / n_cac
any_miss      = miss_bub.any(1)
print(f'fully-called bubbles (0 missing any founder): {(~any_miss).sum():,} / {n_bubbles:,}')

## Headline test: the imbalance inside fully-called bubbles (= pure biology)

Compute the same private-per-founder ratio the composition notebook reported
(14.3× over *all* bubbles), but restricted to bubbles where every founder is
called. There, N-on ≡ N-off, so any imbalance is biology with zero missingness
contribution.

In [ ]:
def priv_ratio(kmask):
    '''cactus-private-per-cactus-founder / PG-private-per-PG-founder over k-mers in kmask.'''
    pmask = kmask & (ac == 1)
    cac_priv = cn[is_cactus][:, pmask].sum() / n_cac
    pg_priv  = cn[is_pg][:, pmask].sum() / n_pg
    return cac_priv, pg_priv, (cac_priv / pg_priv if pg_priv else np.nan)

def sideonly_ratio(kmask):
    '''cactus-only-shared-per-founder / PG-only-shared-per-founder (ac>=2).'''
    s = kmask & (ac >= 2)
    cac_only = s & (ac_p == 0); pg_only = s & (ac_c == 0)
    c = cn[is_cactus][:, cac_only].sum() / n_cac
    p = cn[is_pg][:, pg_only].sum() / n_pg
    return c, p, (c / p if p else np.nan)

kmer_bub_fullycalled = ~any_miss[bubble_id]   # k-mers whose bubble has 0 missing
allk = np.ones(cn.shape[1], dtype=bool)

print('================  PRIVATE k-mers per founder  ================')
for name, km in [('ALL bubbles', allk), ('FULLY-CALLED bubbles only', kmer_bub_fullycalled)]:
    c, p, r = priv_ratio(km)
    print(f'  {name:28s}: cactus/founder={c:8.1f}  PG/founder={p:7.1f}  ratio={r:5.2f}x')
print('================  SIDE-ONLY-SHARED k-mers per founder  ================')
for name, km in [('ALL bubbles', allk), ('FULLY-CALLED bubbles only', kmer_bub_fullycalled)]:
    c, p, r = sideonly_ratio(km)
    print(f'  {name:28s}: cactus/founder={c:8.1f}  PG/founder={p:7.1f}  ratio={r:5.2f}x')

# also: ac==0 fraction in fully-called vs all
print('\nac==0 (dead) fraction:')
print(f'  all bubbles:         {(ac==0).mean()*100:5.2f}%')
print(f'  fully-called bubbles:{(ac[kmer_bub_fullycalled]==0).mean()*100:5.2f}%')

## Dose-response: imbalance vs PG missingness stratum

If missingness *causes* the imbalance, the ratio should climb from ~1× (no PG
missing) up as PG missingness rises. If it's biology, it should be roughly flat
and already large at zero.

In [ ]:
edges = [0.0, 1e-9, 0.02, 0.05, 0.10, 0.20, 0.40, 1.01]
labels = ['0%', '0-2%', '2-5%', '5-10%', '10-20%', '20-40%', '>40%']
bub_stratum = np.digitize(pg_miss_frac, edges[1:-1], right=False)  # 0..len-1

priv_r, side_r, nbub = [], [], []
for s in range(len(labels)):
    bsel = (bub_stratum == s)
    nbub.append(int(bsel.sum()))
    km = bsel[bubble_id]
    priv_r.append(priv_ratio(km)[2])
    side_r.append(sideonly_ratio(km)[2])

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(labels))
ax.plot(x, priv_r, '-o', color='#e31a1c', label='private-k-mer ratio (cactus/PG per founder)')
ax.plot(x, side_r, '-s', color='#1f78b4', label='side-only-shared ratio')
ax.axhline(1.0, color='k', ls='--', lw=1, label='1× = balanced (no imbalance)')
for xi, n in zip(x, nbub):
    ax.annotate(f'{n:,}\nbubbles', (xi, 0.2), ha='center', fontsize=7, color='gray')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('PG missingness in the bubble (fraction of PG founders with any ./.)')
ax.set_ylabel('cactus/PG imbalance ratio')
ax.set_title('Dose-response: does the imbalance grow with PG missingness?')
ax.legend()
fig.tight_layout(); fig.savefig(PLOTDIR/'missingness_doseresponse.png', dpi=130); plt.show()

## Attribution: what share of the cactus private excess lives in missing bubbles?

Of all cactus-carried private k-mers, what fraction sit in bubbles with **any**
PG missingness? If the cactus private excess is a missingness artifact, this
should be near 100%.

In [ ]:
priv = (ac == 1)
cac_priv_km = priv & (ac_c == 1)          # private k-mers carried by a cactus founder
pg_priv_km  = priv & (ac_p == 1)
pgmiss_bub_km = (pg_miss_frac[bubble_id] > 0)

print(f'cactus private k-mers total:           {cac_priv_km.sum():,}')
print(f'  ...in bubbles with >0 PG missing:    {(cac_priv_km & pgmiss_bub_km).sum():,} '
      f'({(cac_priv_km & pgmiss_bub_km).sum()/max(cac_priv_km.sum(),1)*100:.1f}%)')
print(f'PG private k-mers total:               {pg_priv_km.sum():,}')
print(f'  ...in bubbles with >0 PG missing:    {(pg_priv_km & pgmiss_bub_km).sum():,} '
      f'({(pg_priv_km & pgmiss_bub_km).sum()/max(pg_priv_km.sum(),1)*100:.1f}%)')

# share of bubbles that have any PG missing, for reference
print(f'\nreference: {(pg_miss_frac>0).mean()*100:.1f}% of bubbles have >0 PG missing')

## Verdict

Read it off the headline cell + the dose-response curve:

- **Fully-called-bubble ratio ≈ 1×** and the dose-response **climbs from ~1×** →
  the imbalance is **caused by missingness**. The "cactus realizes more rare
  alleles" story is mostly an artifact of PG founders being dropped at
  missing/SV bubbles. → A `cn` missingness fix (mask or dosage) is worth building.
- **Fully-called-bubble ratio still large (≫1×)** and the dose-response **flat** →
  the imbalance is **real biology**. Missingness adds to it but is not the cause.
  → A `cn` fix won't remove the imbalance; filt2 / ω=1/m_b remain the levers.

Whichever way it lands, this is the number to settle *before* implementing
anything. If we want the airtight confirmation, follow with a real N-off Chr1
rebuild and check the all-bubble ratio matches the prediction from this stratified
estimate.